<center>
<img src="https://www.infnet.edu.br/infnet/wp-content/uploads/sites/18/2021/10/infnet-30-horizontal-padrao@300x-8-1024x265.png" width="60%"/>
</center>

# MBA em Engenharia de Dados: Big Data e IA
## Processamento de Big Data com Apache Spark e Spark SQL [26E3_2]
### Projeto da disciplina

## Fonte dos Dados: Instacart Market Basket Analysis

**Origem:** "The Instacart Online Grocery Shopping Dataset 2017", publicado pela Instacart 
em 03/05/2017.

**Artigo de publicação:**
Stanley, J. (2017). *3 Million Instacart Orders, Open Sourced*. Instacart Tech Blog.  
https://tech.instacart.com/3-million-instacart-orders-open-sourced-d40d29ead6f2

**Download:** Atualmente disponível somente via Kaggle (kaggle.com/c/instacart-market-basket-analysis)

Este notebook implementa, em PySpark sobre Databricks, um pipeline de dados seguindo a arquitetura em camadas (medallion architecture — Raw, Bronze, Silver e Gold) para o dataset Instacart Market Basket Analysis: os arquivos CSV originais são baixados do Kaggle e armazenados em volumes na camada Raw, ingeridos sem transformação no formato Delta na camada Bronze, limpos, tipados, padronizados e deduplicados na camada Silver, e por fim combinados e modelados na camada Gold em tabelas prontas para consumo analítico, com a carga incremental entre camadas realizada por meio de operações de merge (upsert) em tabelas Delta.

**Adequação a um cenário de ingestão contínua:** mesmo que este notebook execute uma execução única a partir de do dataset baixado do Kaggle, a arquitetura foi desenhada para suportar um processo de ingestão constante em que novos arquivos chegam periodicamente à camada Raw. Isso é possível porque toda a processamento entre camadas é feito pela função `merge_data`, que realiza um merge Delta baseado nas chaves ou combinação de colunas de cada tabela com a cláusula `whenNotMatchedInsertAll`: registros cuja chave já existe na camada de destino são ignorados, e apenas os registros novos são inseridos. Com isso, o notebook (ou um job agendado que o execute periodicamente) pode ser reprocessado a qualquer momento sobre o conteúdo acumulado da camada Raw sem gerar duplicidade — cada nova chegada de dados brutos é automaticamente propagada, de forma incremental e idempotente, por Bronze, Silver e Gold.

## PREPARAÇÃO DO AMBIENTE

### Download da biblioteca kagglehub para download do dataset

In [0]:
%pip install kagglehub
%restart_python

### Download dos datasets disponíveis no Kaggle: [Instacart Market Basket Analysis](https://www.kaggle.com/datasets/psparks/instacart-market-basket-**analysis**)

In [0]:
import kagglehub

path = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")

print("Caminho para os arquivos baixados:", path)

### Criação dos catálogos, volumes e tabelas que receberão os dados

In [0]:
%sql
-- Para caso seja necessário reexecutar tudo do zero, descomentar a linha abaixo.
-- DROP CATALOG IF EXISTS instacart CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS instacart;

CREATE DATABASE IF NOT EXISTS instacart.raw;

CREATE VOLUME IF NOT EXISTS instacart.raw.aisle;
CREATE VOLUME IF NOT EXISTS instacart.raw.department;
CREATE VOLUME IF NOT EXISTS instacart.raw.product;
CREATE VOLUME IF NOT EXISTS instacart.raw.order;
CREATE VOLUME IF NOT EXISTS instacart.raw.order_product_prior;
CREATE VOLUME IF NOT EXISTS instacart.raw.order_product_train;

### Movendo os dados para os raws volumes

In [0]:
import os
import shutil

catalog_schema = "instacart"

file_to_volume = {
    "aisles.csv": "raw/aisle",
    "departments.csv": "raw/department",
    "products.csv": "raw/product",
    "orders.csv": "raw/order",
    "order_products__prior.csv": "raw/order_product_prior",
    "order_products__train.csv": "raw/order_product_train"
}

for filename in os.listdir(path):
    if filename in file_to_volume:
        volume = file_to_volume[filename]
        src = os.path.join(path, filename)
        dst = f"/Volumes/{catalog_schema}/{volume}/{filename}"
        shutil.copy(src, dst)
        print(f"Copiado: {filename} -> {volume}")
    else:
        print(f"Sem volume correspondente: {filename}")